# EDA 0 — Checkpoint D: payment attempt records and observable journeys (D25–D34)

**Scope:** technical diagnostic of the supplied immutable attempt records. **Not** a payment-outcome definition, causal experiment analysis, measured recovery rate, or financial settlement ledger.

**Order of evidence:** raw attempt → guarded project join → per-project overlapping flags → mutually exclusive fundamental-evidence partition. `created_at` is a **record-creation timestamp**, not the time a failure, authorization, recovery, or Customer Support alert necessarily occurred. The documented experiment affects card handling and observed methods can be post-treatment. All timestamps are parsed as UTC without changing the raw string columns.

This notebook follows the D25–D34 approved inventory subjects: attempts, method diversity, first/last/captured method observability, transitions, timestamp ties, status/method/manual, manual position, recovery-like records, cross-method captures, and diagnostic project classes. The inventory wording below is a **technical paraphrase**, not a verbatim source quotation. All captured-project categories are diagnostic candidates only.

In [1]:
from pathlib import Path
import sys
import csv
from collections import Counter, defaultdict
import pandas as pd
from IPython.display import display
here = Path.cwd().resolve()
ROOT = next((v for v in [here, *here.parents] if (v/'pyproject.toml').is_file() and (v/'data/raw').is_dir()), None)
if ROOT is None: raise FileNotFoundError('Repository root with pyproject.toml and data/raw required')
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.foundation_a import find_project_root, load_raw, assert_conserved
from src.foundation_d import checked_attempts, project_journeys, ordered_record_diagnostics, completion_period
from src.foundation_ce import utc_timestamp

root = find_project_root()
raw, provenance = load_raw(root)
hosts, projects, logs = raw['exposures'], raw['assignments'], raw['payments']
attempts = checked_attempts(projects, logs, hosts)
journeys = project_journeys(projects, attempts)
record_orders, adjacent = ordered_record_diagnostics(attempts)
journeys = journeys.merge(record_orders, on='assignment_id', how='left', validate='one_to_one')
journeys = journeys.merge(hosts[['host_id','experiment_group','first_exposed_at']], on='host_id', how='left', validate='many_to_one')
journeys['completion_period'] = completion_period(journeys['completed_at'])
assert len(journeys) == len(projects)  # source-derived universe; no hardcoded sample size
assert int(journeys['attempts'].sum()) == len(logs)
assert not journeys['experiment_group'].isna().any()
print('Source checksums:')
display(provenance[['file','bytes','sha256']])
print('Registered hosts:',len(hosts),'Completed projects:',len(projects),'Attempt rows:',len(logs))
print('Recorded timestamps parsed as UTC. Original source strings are preserved.')

Source checksums:


,file,bytes,sha256
0,experiment_exposures.csv,481724,9cce67cd8d90d292c0703416b3c28c69783c56d10c2f99...
1,assignments.csv,16411731,c66a6d5cb89b62bc3542ee9184e75c8ded41589e91dac9...
2,payment_logs.csv,24857672,ab6bf97f2ca216504058b5de1deb2f80641d783e8e2365...


Registered hosts: 14708 Completed projects: 279847 Attempt rows: 306957
Recorded timestamps parsed as UTC. Original source strings are preserved.


## D25 — Distribution of attempts per completed project, including zero logs

**Question (approved inventory paraphrase):** How many attempts occur per completed project, including the zero-attempt projects? **Method:** start with all completed projects, merge counts only after validated joins; examine tail and extreme cases. **Limitation:** a log count is not proven retry count. **Status:** answered for recorded attempts.

In [2]:
frequency = journeys['attempts'].value_counts().sort_index().rename_axis('attempts_per_project').reset_index(name='completed_projects')
assert int(frequency['completed_projects'].sum()) == len(projects)
assert int((frequency['attempts_per_project'] * frequency['completed_projects']).sum()) == len(logs)
print('Full low-attempt histogram (remaining bins in tail):')
display(frequency.loc[frequency['attempts_per_project'].le(10)])
print('Distribution summaries:'); display(journeys['attempts'].describe(percentiles=[.5,.9,.95,.99,.999]).to_frame('attempts_per_project'))
print('Projects with 0, 1 or >1 attempts:'); display(pd.Series({'zero':int(journeys.attempts.eq(0).sum()),'one':int(journeys.attempts.eq(1).sum()),'multiple':int(journeys.attempts.gt(1).sum()),'max_attempts':int(journeys.attempts.max())}).to_frame('projects_or_max'))
print('Highest observed counts (technical project IDs, not human narratives):')
display(journeys[['assignment_id','attempts','success_rows','capture_rows','distinct_methods','manual_rows']].sort_values('attempts',ascending=False).head(8))

Full low-attempt histogram (remaining bins in tail):


,attempts_per_project,completed_projects
0,0,239
1,1,265501
2,2,8850
3,3,2511
4,4,1237
5,5,612
6,6,448
7,7,159
8,8,72
9,9,30


Distribution summaries:


,attempts_per_project
count,279847.000000
mean,1.096874
std,0.672132
min,0.000000
50%,1.000000
90%,1.000000
95%,2.000000
99%,3.000000
99.9%,8.000000
max,37.000000


Projects with 0, 1 or >1 attempts:


,projects_or_max
zero,239
one,265501
multiple,14107
max_attempts,37


Highest observed counts (technical project IDs, not human narratives):


,assignment_id,attempts,success_rows,capture_rows,distinct_methods,manual_rows
259659,259660,37,1,1,2,0
243990,243991,32,1,1,1,29
274875,274876,27,1,2,2,3
211850,211851,27,0,2,2,3
256742,256743,27,1,4,2,4
219292,219293,26,1,2,2,2
55130,55131,26,1,2,2,2
17045,17046,26,1,3,2,2


## D26 — Payment methods observed in attempts and projects

**Question:** Which recorded methods appear, and how frequently do projects contain multiple methods? **Method:** count method at attempt grain and project grain separately, and classify observed method sets. **Limitation:** the recorded method is not necessarily the host's independently chosen/pre-treatment method. **Status:** answered for logged methods.

In [3]:
method_attempts = attempts['payment_method'].value_counts(dropna=False).rename('attempt_rows')
method_projects = attempts.groupby('payment_method')['assignment_id'].nunique().rename('distinct_projects_with_method')
method_captures = attempts.loc[attempts['_capture']].groupby('payment_method').agg(capture_field_rows=('payment_log_id','size'),projects_with_capture=('assignment_id','nunique'))
method_table = pd.concat([method_attempts,method_projects,method_captures],axis=1).fillna(0).astype('int64')
assert int(method_table.attempt_rows.sum()) == len(attempts)
assert int(method_table.capture_field_rows.sum()) == int(attempts['_capture'].sum())
display(method_table)
print('Number of recorded methods per project (all projects):')
display(journeys['distinct_methods'].value_counts().sort_index().to_frame('projects'))
# Three documented methods: use a 3-bit vectorized membership mask, no per-project Python UDF.
method_membership = attempts.groupby(['assignment_id','payment_method']).size().unstack(fill_value=0)
method_names = sorted(method_membership.columns.tolist())
method_mask = sum((method_membership[m].gt(0).astype('int64') * (1 << k) for k,m in enumerate(method_names)))
method_labels = {v:' + '.join(m for k,m in enumerate(method_names) if v & (1 << k)) for v in range(1,1<<len(method_names))}
method_sets = method_mask.map(method_labels)
print('Project method sets (overlapping method rows are not additive):')
display(method_sets.value_counts().to_frame('projects'))
assert int(method_sets.value_counts().sum()) == int(journeys.has_logs.sum())

,attempt_rows,distinct_projects_with_method,capture_field_rows,projects_with_capture
payment_method,,,,
card,207031,187012,184616,184408
bank_account,99293,95380,94130,93812
credits,633,631,539,539


Number of recorded methods per project (all projects):


,projects
distinct_methods,
0,239
1,276198
2,3405
3,5


Project method sets (overlapping method rows are not additive):


,projects
card,183751
bank_account,92080
bank_account + card,3146
credits,367
bank_account + credits,149
card + credits,110
bank_account + card + credits,5


## D27 — First/last record-created method and earliest capture-associated method

**Question:** What methods belong to first and last **orderable created records**, and what methods are associated with the first timestamped capture? **Method:** discard projects with tied or unparseable record-creation times from sequence-derived method claims; independently identify earliest captured timestamp and mark first-capture ties. **Limitation:** this does not reveal first/last *financial operation* or actual host choice. **Status:** partially answerable.

In [4]:
ordered_projects = record_orders.loc[record_orders['sequence_order_clear']].copy()
print('First vs last recorded method on projects with unambiguous created_at ordering:')
display(pd.crosstab(ordered_projects['first_record_method'],ordered_projects['last_record_method'],margins=True))
cap = attempts.loc[attempts['_capture'] & attempts['_captured_utc'].notna(),['assignment_id','payment_method','_captured_utc']].copy()
first_cap = cap.groupby('assignment_id')['_captured_utc'].transform('min')
first_cap_rows = cap.loc[cap['_captured_utc'].eq(first_cap)]
first_cap_counts = first_cap_rows.groupby('assignment_id').size()
unique_first = first_cap_rows.loc[first_cap_rows['assignment_id'].isin(first_cap_counts.loc[first_cap_counts.eq(1)].index)]
print('Earliest capture timestamp tied across attempt rows, distinct projects:',int(first_cap_counts.gt(1).sum()))
print('Unique first capture method (only where earliest timestamp has exactly one row):')
display(unique_first['payment_method'].value_counts(dropna=False).to_frame('projects'))
assert int(first_cap_counts.size) <= int(journeys['has_capture_timestamp'].sum())
assert int(unique_first['assignment_id'].nunique()) + int(first_cap_counts.gt(1).sum()) == int(first_cap_counts.size)

First vs last recorded method on projects with unambiguous created_at ordering:


last_record_method,bank_account,card,credits,All
first_record_method,,,,
bank_account,92795,1908,75,94778
card,500,183775,95,184370
credits,76,16,367,459
All,93371,185699,537,279607


Earliest capture timestamp tied across attempt rows, distinct projects: 0
Unique first capture method (only where earliest timestamp has exactly one row):


,projects
payment_method,
card,184217
bank_account,93804
credits,538


## D28 — Observable method sequences, transitions, and returns

**Question:** Which method transitions occur between records ordered by `created_at`? **Method:** adjacent record pairs within a project only when all creation timestamps are valid and unique; method changes counted as *record transitions*. **Limitation:** not proven payment retries or host choice changes. **Status:** observed record transitions answered; financial sequence unresolved.

In [5]:
transitions = adjacent.groupby(['_prev_method','payment_method']).size().rename('adjacent_record_pairs').reset_index()
print('Record-method transitions among orderable projects:'); display(transitions.sort_values('adjacent_record_pairs',ascending=False))
print('Methods changed across adjacent created records:',int(adjacent['_prev_method'].ne(adjacent['payment_method']).sum()))
print('All adjacent record pairs:',len(adjacent))
sequence_count = int(record_orders.loc[record_orders['sequence_order_clear'],'assignment_id'].isin(attempts.assignment_id).sum())
print('Projects with fully orderable records:',sequence_count)
assert int(transitions.adjacent_record_pairs.sum())==len(adjacent)
# Unique method path is a string describing creation-ordered records, not a financial journey.
sequence_view = attempts.loc[attempts.assignment_id.isin(ordered_projects.assignment_id) & attempts.assignment_id.isin(journeys.loc[journeys.attempts.gt(1),'assignment_id'])].sort_values(['assignment_id','_created_utc'])
paths = sequence_view.groupby('assignment_id')['payment_method'].agg(lambda x:' > '.join(x.astype(str)))
print('Most frequent creation-ordered method paths (max 2+ only):')
display(paths.loc[paths.str.contains(' > ',regex=False)].value_counts().head(12).to_frame('projects'))

Record-method transitions among orderable projects:


,_prev_method,payment_method,adjacent_record_pairs
4,card,card,19164
1,bank_account,card,3478
0,bank_account,bank_account,2366
3,card,bank_account,2071
5,card,credits,96
2,bank_account,credits,76
6,credits,bank_account,76
7,credits,card,18
8,credits,credits,2


Methods changed across adjacent created records: 5815
All adjacent record pairs: 27347
Projects with fully orderable records: 279607


Most frequent creation-ordered method paths (max 2+ only):


,projects
payment_method,
card > card,6002
card > card > card,1870
bank_account > card,1638
card > card > card > card,999
bank_account > bank_account,748
card > card > card > card > card,459
bank_account > card > bank_account,414
card > card > card > card > card > card,364
card > bank_account,213


## D29 — Ties and missing creation times; sequence-order limits

**Question:** How many projects lack a unique record-creation ordering? **Method:** report timestamp ties, unparseable/missing timestamps and their union, without synthetic tie-breaking. **Limitation:** even an unambiguous record order does not establish the order of financial events. **Status:** answered for observable ordering.

In [6]:
order_flags = {'projects_with_tied_created_at':int(record_orders['tied_created_timestamp'].sum()),'projects_with_missing_or_invalid_created_at':int(record_orders['missing_or_invalid_created'].sum()),'projects_with_non_unique_order':int((~record_orders['sequence_order_clear']).sum()),'projects_with_orderable_records':int(record_orders['sequence_order_clear'].sum()),'projects_with_no_logs':int((~journeys.has_logs).sum())}
display(pd.Series(order_flags,name='projects').to_frame()); assert_conserved(int(journeys.has_logs.sum()),order_flags['projects_with_non_unique_order'],order_flags['projects_with_orderable_records'])
print('Tied created_at rows by project (if any):')
display(attempts.loc[attempts.duplicated(['assignment_id','_created_utc'],keep=False),['assignment_id','payment_log_id','created_at','payment_status','payment_method']].head(12))
print('Record order alone never proves failure or capture operation time.')

,projects
projects_with_tied_created_at,1
projects_with_missing_or_invalid_created_at,0
projects_with_non_unique_order,1
projects_with_orderable_records,279607
projects_with_no_logs,239


Tied created_at rows by project (if any):


,assignment_id,payment_log_id,created_at,payment_status,payment_method
109076,142500,109077,2025-08-25 20:19:18,voided,bank_account
109077,142500,109078,2025-08-25 20:19:18,declined,card


Record order alone never proves failure or capture operation time.


## D30 — Status by method, manual indicator, and observed group

**Question:** How do raw statuses, timestamped captures, and manual flags distribute across payment methods? **Method:** exhaustive attempt-grain cross-tabs with explicit source totals. **Limitation:** these are attempt descriptions; method and manual activity may be downstream of treatment. **Status:** answered for logged attributes.

In [7]:
method_status_manual = attempts.groupby(['payment_method','payment_status','is_manual_attempt'],dropna=False).agg(attempts=('payment_log_id','size'),timestamped_captures=('_capture','sum'),discordant_rows=('_discordant','sum')).reset_index().sort_values(['payment_method','payment_status','is_manual_attempt'])
display(method_status_manual)
assert int(method_status_manual.attempts.sum()) == len(logs)
assert int(method_status_manual.timestamped_captures.sum()) == int(attempts._capture.sum())
print('Recorded manual attempt rows and projects by group:')
display(attempts.groupby(['experiment_group','is_manual_attempt'],dropna=False).agg(attempts=('payment_log_id','size'),distinct_projects=('assignment_id','nunique')).reset_index())
print('Manual / automatic recorded projects (flags overlap):')
display(pd.Series({'manual':int(journeys.has_manual.sum()),'automatic':int(journeys.has_automatic.sum()),'mixed_manual_auto':int(journeys.mixed_manual_auto.sum())},name='projects').to_frame())

,payment_method,payment_status,is_manual_attempt,attempts,timestamped_captures,discordant_rows
0,bank_account,declined,true,24,0,0
1,bank_account,failed,false,1372,523,523
2,bank_account,failed,true,137,137,137
3,bank_account,succeeded,false,92434,92434,0
4,bank_account,succeeded,true,720,720,0
5,bank_account,voided,false,4599,309,309
6,bank_account,voided,true,7,7,7
7,card,declined,false,12860,40,40
8,card,declined,true,4347,0,0
9,card,failed,true,1,0,0


Recorded manual attempt rows and projects by group:


,experiment_group,is_manual_attempt,attempts,distinct_projects
0,control,false,150829,137501
1,control,true,3603,1966
2,test,false,146766,141912
3,test,true,5759,3382


Manual / automatic recorded projects (flags overlap):


,projects
manual,5348
automatic,279413
mixed_manual_auto,5153


## D31 — Observable position of manual records and September incident

**Question:** Do manually flagged records follow a failure-status record in `created_at` order? **Method:** only projects with complete, unique creation timestamps; separately classify manual-record creation before/during/after September 17–30 UTC. **Limitation:** `is_manual_attempt` is not a Customer Support contact log; creation times are not proven failure or alert times. **Status:** partially answerable; CS behavior and incident effect unidentifiable.

In [8]:
print('Projects with manual records and earlier-created failure-status record (orderable projects only):')
display(pd.Series({'manual_projects_all':int(journeys.has_manual.sum()),'manual_projects_sequence_orderable':int((journeys.has_manual & journeys.sequence_order_clear.astype('boolean').fillna(False)).sum()),'failure_record_created_before_first_manual_record':int(record_orders.failure_record_before_manual_record.sum())},name='projects').to_frame())
incident_start,incident_end=utc_timestamp('2025-09-17'),utc_timestamp('2025-10-01')
created=attempts['_created_utc']
manual_attempts=attempts.loc[attempts['_manual']].copy()
manual_attempts['creation_period'] = 'outside_incident'
manual_attempts.loc[manual_attempts['_created_utc'].lt(incident_start),'creation_period']='before_incident'
manual_attempts.loc[manual_attempts['_created_utc'].ge(incident_start)&manual_attempts['_created_utc'].lt(incident_end),'creation_period']='during_incident'
manual_attempts.loc[manual_attempts['_created_utc'].ge(incident_end),'creation_period']='after_incident'
manual_attempts.loc[manual_attempts['_created_utc'].isna(),'creation_period']='unknown_creation'
manual_incident = manual_attempts.groupby(['experiment_group','creation_period'],dropna=False).agg(manual_attempt_rows=('payment_log_id','size'),distinct_projects=('assignment_id','nunique')).reset_index()
display(manual_incident)
assert int(manual_incident.manual_attempt_rows.sum())==int(attempts._manual.sum())
print('Creation period is NOT a failure/alert/CS contact occurrence time.')

Projects with manual records and earlier-created failure-status record (orderable projects only):


,projects
manual_projects_all,5348
manual_projects_sequence_orderable,5348
failure_record_created_before_first_manual_record,4836


,experiment_group,creation_period,manual_attempt_rows,distinct_projects
0,control,after_incident,1532,817
1,control,before_incident,1717,1004
2,control,during_incident,354,206
3,test,after_incident,2517,1460
4,test,before_incident,2351,1476
5,test,during_incident,891,542


Creation period is NOT a failure/alert/CS contact occurrence time.


## D32 — Failed-status records, later success-status records, and recovery-like patterns

**Question:** Which observed record-status sequences are compatible with follow-up activity after a failure? **Method:** classify co-occurring statuses independently of creation ordering; report failure-status record created before/after success-status record when safe; compare existence of captures separately. **Limitation:** NO financially verified recovery or time-to-recovery is identifiable. **Status:** record patterns answered; recovery not proven.

In [9]:
possible = pd.Series({'projects_with_failure_and_success_status':int(journeys.has_both_failure_and_success.sum()),'projects_with_failure_and_capture_field':int(journeys.has_both_failure_and_capture.sum()),'failure_record_created_before_success_record':int(record_orders.failure_record_before_success_record.sum()),'success_record_created_before_failure_record':int(record_orders.success_record_before_failure_record.sum()),'projects_with_logs_no_success_status':int((journeys.has_logs & ~journeys.has_success_status).sum()),'projects_with_logs_no_capture_field':int((journeys.has_logs & ~journeys.has_capture_timestamp).sum())},name='distinct_projects')
display(possible.to_frame())
print('Attempt status combinations (counts do NOT imply chronological financial outcomes):')
status_membership = attempts.groupby(['assignment_id','payment_status']).size().unstack(fill_value=0)
status_names = sorted(status_membership.columns.tolist())
status_mask = sum((status_membership[k].gt(0).astype('int64') * (1 << i) for i,k in enumerate(status_names)))
status_labels = {v:' + '.join(k for i,k in enumerate(status_names) if v & (1 << i)) for v in range(1,1<<len(status_names))}
status_sets = status_mask.map(status_labels)
display(status_sets.value_counts().head(12).to_frame('projects'))
print('Recorded failure-to-success patterns are NOT verified recovery; status + capture discordances remain as in B12.')

,distinct_projects
projects_with_failure_and_success_status,7125
projects_with_failure_and_capture_field,7248
failure_record_created_before_success_record,7108
success_record_created_before_failure_record,16
projects_with_logs_no_success_status,2317
projects_with_logs_no_capture_field,1049


Attempt status combinations (counts do NOT imply chronological financial outcomes):


,projects
succeeded,264199
succeeded + voided,5967
declined + succeeded,5290
voided,1417
failed + succeeded,938
declined,722
declined + succeeded + voided,533
declined + failed + succeeded,202
declined + failed + succeeded + voided,135
declined + failed,76


Recorded failure-to-success patterns are NOT verified recovery; status + capture discordances remain as in B12.


## D33 — Multiple timestamped captures, multiple methods, and conflicting evidence

**Question:** What configurations of repeated timestamped captures and cross-method capture attempts are observed? **Method:** independently quantify method sets among timestamped capture rows, multiple capture records, concordance, and split-attempt evidence; retain all outlying records. **Limitation:** without amount due, transaction values and operation IDs, duplicate charge, complementary payment and full settlement cannot be distinguished. **Status:** observed fields answered; financial interpretation unidentifiable.

In [10]:
captured_membership = attempts.loc[attempts._capture].groupby(['assignment_id','payment_method']).size().unstack(fill_value=0)
capture_names=sorted(captured_membership.columns.tolist())
capture_mask=sum((captured_membership[m].gt(0).astype('int64') * (1 << k) for k,m in enumerate(capture_names)))
capture_labels={v:' + '.join(m for k,m in enumerate(capture_names) if v & (1 << k)) for v in range(1,1<<len(capture_names))}
capture_sets=capture_mask.map(capture_labels)
print('Capture-associated method sets (project grain):');display(capture_sets.value_counts().to_frame('distinct_projects'))
flags={ 'projects_with_2plus_succeeded_attempts':int(journeys.success_rows.ge(2).sum()),'projects_with_2plus_capture_rows':int(journeys.multiple_capture_rows.sum()),'projects_with_multiple_distinct_capture_timestamps':int(journeys.distinct_capture_timestamps.gt(1).sum()),'projects_with_2plus_capture_methods':int(journeys.multiple_capture_methods.sum()),'projects_with_any_discordant_attempt':int(journeys.has_discordance.sum()),'projects_with_both_evidence_on_different_attempts_only':int(journeys.split_evidence_attempts_only.sum()),'projects_with_multiple_methods_any_attempt':int(journeys.multiple_methods.sum()) }
display(pd.Series(flags,name='distinct_projects').to_frame());assert int(journeys.capture_rows.sum())==int(attempts._capture.sum())
print('Example configurations only (no financial attribution):')
display(journeys.loc[journeys.multiple_capture_methods,['assignment_id','attempts','capture_rows','success_rows','discordant_rows','capture_methods','distinct_methods']].head(8))

Capture-associated method sets (project grain):


,distinct_projects
card,184208
bank_account,93614
credits,537
bank_account + card,198
card + credits,2


,distinct_projects
projects_with_2plus_succeeded_attempts,154
projects_with_2plus_capture_rows,652
projects_with_multiple_distinct_capture_timestamps,652
projects_with_2plus_capture_methods,200
projects_with_any_discordant_attempt,2075
projects_with_both_evidence_on_different_attempts_only,140
projects_with_multiple_methods_any_attempt,3410


Example configurations only (no financial attribution):


,assignment_id,attempts,capture_rows,success_rows,discordant_rows,capture_methods,distinct_methods
959,960,2,2,1,1,2,2
4385,4386,2,2,2,0,2,2
5067,5068,2,2,1,1,2,2
6646,6647,5,2,1,1,2,2
7896,7897,8,3,1,2,2,2
8245,8246,2,2,1,1,2,2
8888,8889,3,2,1,1,2,2
11490,11491,2,2,1,1,2,2


## D34 — Exclusive fundamental evidence classes, overlapping flags, and measurement sensitivity

**Question:** What journey/evidence classes are technically defensible and what remains ambiguous? **Method:** create an **exclusive five-bucket evidence partition** and a separate overlapping flag inventory; compare three candidate capture evidence rules with a group × completion-period denominator, never selecting an experiment KPI. **Limitation:** monetary settlement, censored late captures, log gaps, and operation truth remain unidentifiable. **Status:** evidence observable; financial outcomes partially identifiable at best.

In [11]:
part = journeys.financial_evidence_partition.value_counts().rename_axis('exclusive_fundamental_evidence').reset_index(name='completed_projects')
display(part)
assert int(part.completed_projects.sum())==len(projects)
overlap_cols=['has_logs','has_success_status','has_capture_timestamp','has_concordant_attempt','split_evidence_attempts_only','has_discordance','multiple_capture_rows','multiple_capture_methods','multiple_methods','has_manual','mixed_manual_auto','has_both_failure_and_success','failure_record_before_success_record','materially_ambiguous_record_evidence']
overlap_counts=journeys[overlap_cols].astype('boolean').fillna(False).sum().astype(int).rename('projects; overlapping, never additive')
display(overlap_counts.to_frame())
print('The three alternative candidate rules refer to any attempt with succeeded, any nonnull captured_at, and any same-attempt concordance.')
candidate_cols=['has_success_status','has_capture_timestamp','has_concordant_attempt']
rule_patterns=journeys.groupby(candidate_cols,dropna=False).size().rename('completed_projects').reset_index()
display(rule_patterns)
assert int(rule_patterns.completed_projects.sum())==len(projects)
print('Projects changing binary classification between candidate rules:')
display(pd.Series({'success_xor_capture':int((journeys.has_success_status ^ journeys.has_capture_timestamp).sum()),'success_xor_concordant':int((journeys.has_success_status ^ journeys.has_concordant_attempt).sum()),'capture_xor_concordant':int((journeys.has_capture_timestamp ^ journeys.has_concordant_attempt).sum())},name='projects').to_frame())
print('Project counts / denominators by recorded assignment group and COMPLETION period (not treatment effects):')
by_group=journeys.groupby(['experiment_group','completion_period'],dropna=False).agg(completed_projects=('assignment_id','size'),candidate_status_projects=('has_success_status','sum'),candidate_timestamp_projects=('has_capture_timestamp','sum'),candidate_concordant_projects=('has_concordant_attempt','sum'),no_logs=('has_logs',lambda s:int((~s).sum()))).reset_index()
for name in ['candidate_status_projects','candidate_timestamp_projects','candidate_concordant_projects']:
    by_group[name+'_pct_of_completed']=(100*by_group[name]/by_group['completed_projects']).round(3)
display(by_group)
assert int(by_group.completed_projects.sum())==len(projects)
for k,c in [('candidate_status_projects','has_success_status'),('candidate_timestamp_projects','has_capture_timestamp'),('candidate_concordant_projects','has_concordant_attempt')]:
    assert int(by_group[k].sum())==int(journeys[c].sum())
print('These percentages are measurement coverage under candidate rules, NOT approved payment capture rates.')

,exclusive_fundamental_evidence,completed_projects
0,both_evidence_types,277255
1,capture_timestamp_only,1304
2,logs_no_success_or_capture,1013
3,no_logs,239
4,success_status_only,36


,"projects; overlapping, never additive"
has_logs,279608
has_success_status,277291
has_capture_timestamp,278559
has_concordant_attempt,277115
split_evidence_attempts_only,140
has_discordance,2075
multiple_capture_rows,652
multiple_capture_methods,200
multiple_methods,3410
has_manual,5348


The three alternative candidate rules refer to any attempt with succeeded, any nonnull captured_at, and any same-attempt concordance.


,has_success_status,has_capture_timestamp,has_concordant_attempt,completed_projects
0,False,False,False,1252
1,False,True,False,1304
2,True,False,False,36
3,True,True,False,140
4,True,True,True,277115


Projects changing binary classification between candidate rules:


,projects
success_xor_capture,1340
success_xor_concordant,176
capture_xor_concordant,1444


Project counts / denominators by recorded assignment group and COMPLETION period (not treatment effects):


,experiment_group,completion_period,completed_projects,candidate_status_projects,candidate_timestamp_projects,candidate_concordant_projects,no_logs,candidate_status_projects_pct_of_completed,candidate_timestamp_projects_pct_of_completed,candidate_concordant_projects_pct_of_completed
0,control,baseline,38731,38473,38638,38456,21,99.334,99.760,99.290
1,control,experiment,98855,97948,98405,97880,35,99.082,99.545,99.014
2,test,baseline,39814,39498,39618,39460,58,99.206,99.508,99.111
3,test,experiment,102447,101372,101898,101319,125,98.951,99.464,98.899


These percentages are measurement coverage under candidate rules, NOT approved payment capture rates.


## Technical synthesis — implications for later methodology (NOT an approval)

**Directly observed:** payment-attempt records, status/method/manual fields, record creation, populated capture timestamps, project association, exposure-file group labels, and completed-project dates. All projects, including zero-log projects, remain in the diagnostic population.

**Recovery-compatible, not verified:** a failed-status **record created** before another success-status or manual record; mixed statuses or multiple capture timestamps. Actual failure, recovery and Customer Support timestamps are not observed. **Overlapping flags are not summed**.

**Material choices left for Fernando/Jurandir/Adelaide:** which of the three candidate capture evidences (if any) to use; follow-up windows and censoring (Checkpoint C); population/exposure definition (Checkpoint E); treatment-sensitive post-exposure method selection; interpretations of no logs and divergent statuses. These unresolved choices affect would-be project/host counts and denominators.

**Impossible from these three CSVs:** amounts due/collected, full settlement, financial duplicate charges, authorization/reauthorization events, complete CS interventions, timing of true failures, causal impact or Ship/Kill decisions. The A1 operational premise remains provisional. No hypotheses or treatment-effect comparisons are tested.

**V7:** completion status of this exact notebook is recorded separately after a fresh-kernel run. A/B's historical clean-kernel V7 exceptions remain unresolved; C/E have passed in the authoring environment.